# m10 lab · Arrhythmia mechanisms

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Three mechanisms recur in mapping: a **focus** that fires and spreads outward, **reentry** around a circuit, and
**rotors**, spiral waves turning around a phase singularity. You compute the wavelength, find the shortest ring that
sustains reentry, watch alternans appear when restitution is steep, detect a spiral's singularity, and compare the
activation span of focal and reentrant patterns.

| Part | What you do | Equation |
| --- | --- | --- |
| A | The wavelength of excitation | eq-m10-wavelength |
| B | Reentry on a ring: the shortest circuit | eq-m10-circuit |
| C | Restitution and alternans | eq-m10-restitution, eq-m10-alternans |
| D | A spiral wave and its phase singularity | eq-m29-charge |
| E | Focal or reentrant? The activation span | eq-m10-focal |

Cell kinetics are the m08 Mitchell–Schaeffer model; the spiral uses Barkley's two-variable model with illustrative,
dimensionless parameters. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import arrhythmia as A
from epml.membrane import MS_PARAMS as P

## Part A · The wavelength

A wave occupies a stretch of tissue that is still refractory: its **wavelength** $\lambda = \text{CV}\times\text{ERP}$
(eq-m10-wavelength). In conscious dogs Rensma and colleagues found 14–18 cm during normal rhythm, atrial flutter below
9.7 cm and fibrillation below 7.8 cm (1988). Compare with the m08 and m09 model numbers: refractory period 217 ms at
500-ms pacing, 153 ms with the shortened action potential, and 1.0 m/s.

In [ ]:
for label, erp in (("m08 model", 217.0), ("shorter action potential (m08 part E)", 153.0)):
    for cv in (1.0, 0.6):
        print(f"{label}, {cv} m/s: wavelength {A.wavelength(cv, erp) / 10:.1f} cm")
lam_short = A.wavelength(0.6, 153.0)

In [ ]:
# Checks
assert abs(A.wavelength(1.0, 217.0) - 217.0) < 1e-12
assert lam_short < 97.0                                      # below the flutter value Rensma et al. report

## Part B · Reentry on a ring

A fixed circuit of length $L$ supports reentry only if it is longer than the wavelength; the cycle length is $L/\text{CV}$
and the **excitable gap** is $L - \lambda$ (eq-m10-circuit). Start a Mitchell–Schaeffer ring ($D$ = 0.1 mm²/ms, about
0.34 m/s on a long cable, m09) with a fully developed pulse and shrink the ring. About 30 s.

In [ ]:
pulse = A.traveling_pulse()
v, h, front = pulse
tail = (front - np.max(np.where(v[:front] < P["v_gate"])[0])) * 0.5
print(f"pulse: tissue above v_gate extends {tail:.1f} mm behind the front")
rows = []
for L in (70, 80, 90, 100, 120, 160, 200):
    tt = A.ring_reentry(float(L), t_end=2500.0, pulse=pulse)
    cl = np.diff(tt)[-1] if len(tt) > 1 else np.nan
    rows.append((L, len(tt), cl, L / cl if np.isfinite(cl) else np.nan))
    print(f"ring {L:3d} mm: {'sustained' if len(tt) > 1 else 'failed   '}"
          + (f", cycle length {cl:.0f} ms, velocity {L / cl:.3f} m/s" if np.isfinite(cl) else ""))
ok = [r for r in rows if np.isfinite(r[2])]
plt.plot([r[0] for r in ok], [r[2] for r in ok], "o-"); plt.xlabel("ring length (mm)"); plt.ylabel("cycle length (ms)"); plt.show()

The shortest ring that sustains reentry (90 mm here) is a little longer than the pulse's tail. On short rings the
front travels into tissue that has had little time to recover, so it slows: velocity restitution. Real circuits add
anatomy, curvature and heterogeneity.

In [ ]:
# Checks
sustained = {r[0]: r[1] > 1 for r in rows}
assert not sustained[80] and sustained[90] and sustained[200]
assert ok[0][3] < ok[-1][3]                                # shorter rings conduct more slowly

## Part C · Restitution and alternans

Pace at cycle length CL and each APD follows from the previous one, $\text{APD}_{n+1} = f(\text{CL} - \text{APD}_n)$
(eq-m10-restitution). A disturbance is multiplied by $-f'$ each beat, so it alternates in sign and grows when the
slope exceeds 1. Nolasco and Dahlen introduced this graphic method (1968). With the m08 closed form the slope reaches 1
at a computable cycle length (eq-m10-alternans); compare with a paced model cell.

In [ ]:
di_c, apd_c, cl_c = A.alternans_onset()
print(f"closed-form map: slope 1 at DI = {di_c:.1f} ms, APD = {apd_c:.1f} ms, CL = {cl_c:.1f} ms")
for cl in (200.0, 160.0, 150.0):
    apd, di = A.fixed_point(cl)
    seq = A.iterate_map(cl, apd + 2, 30)
    print(f"map at CL {cl:.0f}: slope {A.restitution_slope(di):.2f}, last two APDs {seq[-2]:.1f}, {seq[-1]:.1f}")
sim = {}
for cl in (300.0, 230.0, 210.0, 190.0):
    a = A.paced_apds(cl, 40)
    sim[cl] = a
    print(f"paced cell at CL {cl:.0f}: {len(a)} of 40 beats captured, last APDs {np.round(a[-4:], 1)}")
for cl, a in sim.items():
    plt.plot(a, ".-", label=f"CL {cl:.0f} ms")
plt.xlabel("beat"); plt.ylabel("APD (ms)"); plt.legend(); plt.show()

The model cell alternates at 210 ms and drops to 2:1 capture at 190 ms, at longer cycle lengths than the closed-form
map predicts, because its APDs run longer than the closed form (m08). The mechanism is the same: steep restitution.

In [ ]:
# Checks
assert abs(A.restitution_slope(di_c) - 1) < 1e-12
assert np.ptp(sim[300.0][-6:]) < 0.5
assert abs(sim[210.0][-1] - sim[210.0][-2]) > 5
assert len(sim[190.0]) < 30                                 # 2:1 block

## Part D · A spiral wave and its phase singularity

Break a wavefront and its free end curls into a spiral. In Barkley's two-variable model (illustrative parameters,
dimensionless units), cross-field initial conditions do this at the center of the sheet. Compute each point's phase in
the $(u, v)$ plane and count the topological charge around every grid square (eq-m29-charge).

In [ ]:
u, vv = A.barkley_spiral(n=120, t_end=60.0)
phase = A.phase_field(u, vv)
ps = A.singularities(phase)
print("phase singularities (row, column, charge):", ps.tolist())
ps_mirror = A.singularities(A.phase_field(u[:, ::-1], vv[:, ::-1]))
print("mirror image:", ps_mirror.tolist())
fig, ax = plt.subplots(1, 2, figsize=(8, 3.8))
ax[0].imshow(u, origin="lower", cmap="gray"); ax[0].set_title("u")
ax[1].imshow(phase, origin="lower", cmap="hsv"); ax[1].plot(ps[:, 1] + 0.5, ps[:, 0] + 0.5, "k*", ms=12)
ax[1].set_title("phase and singularity"); plt.show()

In [ ]:
# Checks
assert len(ps) == 1 and abs(ps[0, 2]) == 1
assert len(ps_mirror) == 1 and ps_mirror[0, 2] == -ps[0, 2]

## Part E · Focal or reentrant?

From a focus, activation spreads outward and the map covers only the time needed to reach the farthest point
(eq-m10-focal). Around a circuit, activation runs continuously and covers the whole cycle; the earliest and latest
sites sit next to each other. Compare the two in a 40 × 40 mm area.

In [ ]:
y, x = np.mgrid[-20:21, -20:21].astype(float)
t_focal = A.focal_activation(x, y, 5.0, -3.0, 1.0)            # focus inside the area, 1.0 m/s
span_focal = A.activation_span(t_focal, 300.0)
theta = np.linspace(0, 2 * np.pi, 361)[:-1]
r_ring, cv_ring = 15.0, 0.5
t_ring = theta * r_ring / cv_ring
cl_ring = 2 * np.pi * r_ring / cv_ring
span_ring = A.activation_span(t_ring, cl_ring)
print(f"focal, CL 300 ms: activation spans {100 * span_focal:.0f}% of the cycle")
print(f"circuit of radius {r_ring} mm at {cv_ring} m/s (CL {cl_ring:.0f} ms): activation spans {100 * span_ring:.0f}% of the cycle")

In [ ]:
# Checks
assert span_focal < 0.2 and span_ring > 0.99

## Where to go next

- In part B, lower $D$ to slow conduction: how does the shortest ring change?
- In part C, change $\tau_{\text{close}}$ (m08 part E) and find the new alternans threshold.
- In part D, run longer, or add a second broken front, and track how many singularities appear and annihilate.
- m29 measures the same singularities on catheter recordings, where electrode spacing limits what can be seen.